Your workflow is **excellent and well-structured**! Here are the step-wise prompts for your Copilot agent, with a few additions to ensure completeness:

---

## Complete Optimization Workflow - Copilot Agent Prompts

### **Phase 0: Setup & Data Preparation**



In [ ]:
PROMPT 0.1: "Create a configuration file or class that defines:
- List of asset classes and specific assets to train
- Cross-asset feature groups (e.g., {'SPY': ['commodity', 'int_equity']})
- Date ranges for train/validation/test splits
- Default RF hyperparameters from the paper
- Transaction cost assumptions (spread + market impact)
Save this as a reusable config that can be easily modified."

In [ ]:
PROMPT 0.2: "Add a method to KMRF class called `load_asset_price_data()` that:
- Loads price data for the target asset
- Calculates returns for backtesting
- Aligns dates with feature data and labels
- Returns a DataFrame with columns: ['close', 'returns', 'log_returns']
This will be used for single-asset strategy backtesting."





---

### **Phase 1: Initial Training with Baseline Settings**



In [ ]:
PROMPT 1.1: "Update the KMRF initialization in test_kmrf_new.ipynb to use:
- feature_window_size = 1 (single-step features, matching the paper)
- feature_asset_classes = ['commodity'] (or whatever cross-asset judgment you made)
- classification_type = 'original' (4-regime labels)
- Default RF hyperparameters from paper (already in place)
Document the rationale for cross-asset feature selection in a markdown cell."

In [ ]:
PROMPT 1.2: "Run the complete training pipeline:
1. Load data and features
2. Prepare training data with cross-asset features
3. Train initial Random Forest model
4. Run consensus feature selection with min_votes=2
5. Retrain with selected features
6. Save the selected feature list to a file for reproducibility
Document the number of features before/after selection and top features."





---

### **Phase 2: Single-Asset Strategy Implementation**



In [ ]:
PROMPT 2.1: "Create a new class `SingleAssetKMRFStrategy` in a file called 'single_asset_strategy.py' with methods:
- __init__(kmrf_model, asset_data, transaction_costs)
- generate_signals(regime_probs, strategy_params) → returns position series
- backtest(signals, returns) → returns performance metrics
- calculate_metrics(strategy_returns, benchmark_returns) → dict of Sharpe, Sortino, max_drawdown, etc.

Include different signal generation strategies:
1. 'threshold': Long if P(Bullish) > threshold, Cash if P(Bearish) > threshold
2. 'proportional': Position = P(Bullish) - P(Bearish)
3. 'regime_specific': Different allocations per regime (0,1,2,3)

Transaction costs should include:
- Fixed spread (e.g., 2 bps)
- Market impact proportional to trade size
Make strategy_params a dict that can include signal thresholds and position sizing rules."

In [ ]:
PROMPT 2.2: "Add a method `estimate_transaction_costs()` to the strategy class that:
- Takes position changes as input
- Calculates total costs as: spread + impact(trade_size)
- Uses realistic assumptions: 2-5 bps spread for liquid ETFs, 10-20 bps for commodities
- Includes slippage based on volatility
- Returns cost per trade and total cost over backtest period
This should be called within the backtest() method."





---

### **Phase 3: Hyperparameter Optimization Setup**



In [ ]:
PROMPT 3.1: "Create a new notebook 'optimize_single_asset.ipynb' that:
1. Imports the KMRF class and SingleAssetKMRFStrategy
2. Loads a single asset (start with SPY)
3. Sets up Optuna optimization framework
4. Defines the search space for hyperparameters (see next prompt)
5. Optimizes on validation set using Sortino ratio as objective
6. Saves best hyperparameters and validation metrics
7. Evaluates final model on test set with best hyperparameters"

In [ ]:
PROMPT 3.2: "Define the hyperparameter search space for Optuna optimization. Include:

RF Hyperparameters:
- n_estimators: Integer [100, 500]
- max_depth: Integer [5, 20]
- min_samples_split: Integer [20, 200]
- min_samples_leaf: Integer [20, 200]
- max_samples: Float [0.3, 0.9]
- max_features: Float [0.1, 0.5]
- min_weight_fraction_leaf: Float [0.0, 0.1]

Feature Window:
- feature_window_size: Integer [1, 10]

Strategy Parameters:
- signal_strategy: Categorical ['threshold', 'proportional', 'regime_specific']
- bull_threshold: Float [0.5, 0.8] (if threshold strategy)
- bear_threshold: Float [0.5, 0.8] (if threshold strategy)
- position_scaling: Float [0.5, 1.5] (if proportional strategy)

Feature Selection:
- min_votes: Integer [2, 3] (for consensus feature selection)
- cumulative_importance_threshold: Float [0.85, 0.98]
- mi_top_pct: Float [0.2, 0.5]

Create a function `create_optuna_study()` that returns a configured study object."

In [ ]:
PROMPT 3.3: "Implement the Optuna objective function that:
1. Receives a trial object with suggested hyperparameters
2. Re-initializes KMRF model with trial's RF hyperparameters
3. Re-trains on training data with trial's feature_window_size
4. Runs consensus feature selection with trial's selection parameters
5. Generates regime predictions on validation set
6. Runs single-asset strategy with trial's strategy parameters
7. Calculates Sortino ratio on validation set
8. Returns negative Sortino (Optuna minimizes by default) or positive Sortino (if maximizing)

Include error handling and logging of failed trials."







---

### **Phase 4: Additional Optimizations (Missing Parameters)**



In [ ]:
PROMPT 4.1: "Add optimization for class weights in Random Forest. Many assets may have imbalanced regime distributions. Include:
- class_weight: Categorical ['balanced', 'balanced_subsample', None]
This helps the model not just predict the most common regime."

In [ ]:
PROMPT 4.2: "Add optimization for the consensus feature selection variance threshold:
- variance_threshold: Float [0.001, 0.05]
Very low-variance features may still contain signal but create noise. Test different thresholds."

In [ ]:
PROMPT 4.3: "Add optimization for position sizing constraints:
- max_position_size: Float [0.5, 1.0] (maximum allocation)
- min_position_size: Float [0.0, 0.2] (minimum allocation when signal is active)
This helps manage risk and prevents over-concentration."

In [ ]:
PROMPT 4.4: "Add optimization for rebalancing frequency:
- rebalance_frequency: Integer [1, 20] (days between rebalances)
More frequent rebalancing may capture signals faster but increases costs. Less frequent reduces costs but may miss regime changes.

Modify the backtest() method to only allow position changes every N days."

In [ ]:
PROMPT 4.5: "Add optimization for stop-loss and take-profit rules:
- stop_loss_pct: Float [0.0, 0.1] (0 = no stop loss)
- take_profit_pct: Float [0.0, 0.2] (0 = no take profit)

These exit rules can help manage tail risk. Implement in the backtest() method to exit positions when thresholds are hit, regardless of regime predictions."











---

### **Phase 5: Cross-Asset Feature Re-Evaluation**



In [ ]:
PROMPT 5.1: "After initial optimization on SPY, create a method to test different cross-asset feature combinations:
- Test: no cross-asset features (baseline)
- Test: commodity features only
- Test: international equity features only  
- Test: both commodity + international equity
- Test: adding US Treasury features

For each combination, run a quick optimization (50 trials) and compare validation Sortino ratios. This validates your initial judgment about which cross-asset features to include."

In [ ]:
PROMPT 5.2: "Create a visualization showing:
- Heatmap of Sortino ratios for different cross-asset feature combinations
- Feature importance rankings from the best model
- Which specific cross-asset features appear most frequently in top models

This helps decide if cross-asset features are worth the added complexity and training time."





---

### **Phase 6: Batch Optimization Across Assets**



In [ ]:
PROMPT 6.1: "Create a batch optimization script that:
- Loops through all assets in your universe
- For each asset, runs Optuna optimization (suggest 100-200 trials per asset)
- Saves best hyperparameters to a JSON file
- Saves validation and test performance metrics
- Creates a summary DataFrame comparing all assets

Use parallel processing (Optuna's built-in parallelization) to speed this up."

In [ ]:
PROMPT 6.2: "Add early stopping to the optimization:
- If no improvement in top 10 trials for 20 consecutive trials, stop
- If validation Sortino < 0.5 after 50 trials, mark asset as 'unpredictable' and skip
This saves computation time on assets where KMRF doesn't work well."





---

### **Phase 7: Robustness Checks**



In [ ]:
PROMPT 7.1: "Implement walk-forward analysis:
- Instead of single train/val/test split, use rolling windows
- Train on years 1995-2015, validate on 2016-2017, test on 2018
- Re-train on 1995-2016, validate on 2017-2018, test on 2019
- Continue through all available data
- Check if optimal hyperparameters are stable across time periods

If hyperparameters vary drastically, the optimization may be overfitting."

In [ ]:
PROMPT 7.2: "Add Monte Carlo simulation to test parameter sensitivity:
- For the best hyperparameters found, randomly perturb each by ±10%
- Re-run backtest 100 times with perturbed parameters
- Calculate distribution of Sortino ratios
- If std(Sortino) is high, the model is fragile and needs regularization

Report mean, std, and 5th/95th percentiles of performance metrics."





---

### **Phase 8: Final Model Selection & Documentation**



In [ ]:
PROMPT 8.1: "Create a final summary notebook that shows:
- Table of best hyperparameters for each asset
- Performance metrics (Sharpe, Sortino, max drawdown, win rate) for each asset
- Assets ranked by validation Sortino ratio
- Recommendation on which assets to include in MPC portfolio (e.g., top 10-15)
- Comparison of your optimized model vs. paper's default hyperparameters

Include visualizations of equity curves and drawdowns."

In [ ]:
PROMPT 8.2: "Save the final optimized models:
- For each asset, save the best KMRF model (with optimal hyperparameters)
- Save the selected features list
- Save the optimal strategy parameters
- Create a metadata file documenting the optimization process and results

Use a consistent naming convention: '{asset_name}_optimized_model_{date}.pkl'"

In [ ]:
PROMPT 8.3: "Document the entire optimization process in a markdown file 'OPTIMIZATION_METHODOLOGY.md' including:
- Rationale for hyperparameter search ranges
- Justification for using Sortino ratio vs. Sharpe
- How transaction costs were estimated
- Why certain assets were excluded
- Lessons learned and recommendations for future improvements

This documentation is critical for reproducibility and explaining your methodology."







---

## Summary of Missing Parameters You Should Optimize

Beyond your original list, also optimize:

1. ✅ **Class Weights** - Handle imbalanced regime distributions
2. ✅ **Variance Threshold** - Fine-tune feature selection sensitivity  
3. ✅ **Position Sizing Constraints** - Min/max allocation limits
4. ✅ **Rebalancing Frequency** - Balance signal capture vs. transaction costs
5. ✅ **Stop-Loss / Take-Profit** - Tail risk management
6. ✅ **Cross-Asset Feature Combinations** - Validate your initial judgment
7. ✅ **Early Stopping Criteria** - Save computation time

---

## Recommended Optimization Order

1. **Quick test (50 trials)** on SPY with default cross-asset features
2. **Cross-asset feature ablation** (test different combinations)
3. **Full optimization (200 trials)** on SPY with best cross-asset setup
4. **Robustness checks** (walk-forward, Monte Carlo) on SPY
5. **Batch optimization** across all assets using learned search ranges
6. **Final selection** of top 10-15 assets for MPC portfolio

This workflow ensures you don't waste compute on unpromising assets and validates assumptions before scaling up. 🎯